In [ ]:
import os
import sys

from pyspark.sql import SparkSession

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .appName("PySparkDemo")
    .master("local[*]")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
print("SparkSession created successfully.")

In [ ]:
data = [
    (101, "  Amit Sharma  ", "IT", 75000.50, "2024-01-15", "Mumbai"),
    (102, "Neha Patel", "HR", 55000.75, "2023-06-20", "Pune"),
    (103, "Rahul Verma", "IT", 80000.00, "2022-03-10", "Delhi"),
    (104, "Priya Singh", "Finance", 65000.25, "2024-08-05", "Mumbai"),
    (105, "Ravi Kumar", "IT", 72000.00, "2023-11-12", "Pune"),
    (106, "Sneha Shah", "HR", None, "2024-02-18", None)
]

columns = [
    "EmpID",
    "Name",
    "Department",
    "Salary",
    "JoiningDate",
    "City"
]

df = spark.createDataFrame(data, columns)

df.show(truncate=False)
df.printSchema()

In [ ]:
# String Functions
from pyspark.sql import functions as F

string_df = df.select(
    "Name",
    F.upper(F.col("Name")).alias("UpperName"),
    F.lower(F.col("Name")).alias("LowerName"),
    F.initcap(F.col("Name")).alias("ProperName")
)

string_df.show(truncate=False)

In [ ]:
trim_df = df.select(
    "Name",
    F.trim(F.col("Name")).alias("TrimmedName"),
    F.ltrim(F.col("Name")).alias("LeftTrimmed"),
    F.rtrim(F.col("Name")).alias("RightTrimmed")
)

trim_df.show(truncate=False)

In [ ]:
result_df = df.select(
    F.trim("Name").alias("Name"),
    F.length(F.trim("Name")).alias("NameLength"),
    F.substring(F.trim("Name"), 1, 4).alias("FirstFourChars")
)

result_df.show(truncate=False)

In [ ]:
concat_df = df.select(
    "EmpID",
    F.concat(
        F.lit("EMP-"),
        F.col("EmpID").cast("string")
    ).alias("EmployeeCode"),
    F.concat_ws(
        " - ",
        F.trim("Name"),
        F.col("Department")
    ).alias("EmployeeDetails")
)

concat_df.show(truncate=False)

In [ ]:
names_data = [
    (1, "Amit   Sharma"),
    (2, "Neha     Patel"),
    (3, "Rahul  Verma")
]

names_df = spark.createDataFrame(
    names_data,
    ["ID", "FullName"]
)

clean_names_df = (
    names_df
    .withColumn(
        "CleanName",
        F.regexp_replace(F.col("FullName"), r"\s+", " ")
    )
    .withColumn(
        "FirstName",
        F.split(F.col("CleanName"), " ")[0]
    )
    .withColumn(
        "LastName",
        F.split(F.col("CleanName"), " ")[1]
    )
)

clean_names_df.show(truncate=False)

In [ ]:
# Date & Timestamp Functions
date_df = df.withColumn(
    "JoiningDate",
    F.to_date(F.col("JoiningDate"), "yyyy-MM-dd")
)

date_df.printSchema()
date_df.show()

In [ ]:
date_parts_df = date_df.select(
    "Name",
    "JoiningDate",
    F.year("JoiningDate").alias("JoiningYear"),
    F.month("JoiningDate").alias("JoiningMonth"),
    F.dayofmonth("JoiningDate").alias("JoiningDay")
)

date_parts_df.show(truncate=False)

In [ ]:
current_df = df.select(
    "EmpID",
    F.current_date().alias("Today"),
    F.current_timestamp().alias("CurrentTimestamp")
)

current_df.show(truncate=False)

In [ ]:
experience_df = date_df.withColumn(
    "ExperienceYears",
    F.round(
        F.months_between(
            F.current_date(),
            F.col("JoiningDate")
        ) / 12,
        2
    )
)

experience_df.select(
    "Name",
    "JoiningDate",
    "ExperienceYears"
).show()

In [ ]:
date_calculation_df = date_df.select(
    "Name",
    "JoiningDate",
    F.date_add("JoiningDate", 30).alias("After30Days"),
    F.date_sub("JoiningDate", 7).alias("Before7Days"),
    F.add_months("JoiningDate", 6).alias("After6Months")
)

date_calculation_df.show(truncate=False)

In [ ]:
date_difference_df = date_df.withColumn(
    "DaysSinceJoining",
    F.datediff(
        F.current_date(),
        F.col("JoiningDate")
    )
)

date_difference_df.show()

In [ ]:
timestamp_data = [
    (1, "2026-11-20 10:30:45"),
    (2, "2026-11-20 14:15:20"),
    (3, "2026-11-21 09:05:10")
]

timestamp_df = spark.createDataFrame(
    timestamp_data,
    ["EventID", "EventTime"]
)

timestamp_df = timestamp_df.withColumn(
    "EventTimestamp",
    F.to_timestamp("EventTime", "yyyy-MM-dd HH:mm:ss")
)

timestamp_df.select(
    "EventID",
    "EventTimestamp",
    F.hour("EventTimestamp").alias("Hour"),
    F.minute("EventTimestamp").alias("Minute"),
    F.second("EventTimestamp").alias("Second"),
    F.date_format(
        "EventTimestamp",
        "dd-MM-yyyy"
    ).alias("FormattedDate")
).show(truncate=False)

In [ ]:
# Mathematical Functions
math_data = [
    (1, 125.678),
    (2, 89.234),
    (3, -45.789)
]

math_df = spark.createDataFrame(
    math_data,
    ["ID", "Amount"]
)

result_df = math_df.select(
    "Amount",
    F.round("Amount", 2).alias("Rounded"),
    F.ceil("Amount").alias("Ceiling"),
    F.floor("Amount").alias("Floor"),
    F.abs("Amount").alias("Absolute")
)

result_df.show()

In [ ]:
numbers_df = spark.createDataFrame(
    [(1, 16.0), (2, 25.0), (3, 36.0)],
    ["ID", "Number"]
)

numbers_df.select(
    "Number",
    F.sqrt("Number").alias("SquareRoot"),
    F.pow("Number", 2).alias("Square")
).show()

In [ ]:
increment_df = df.select(
    "Name",
    "Salary",
    F.round(
        F.col("Salary") * 0.10,
        2
    ).alias("Increment"),
    F.round(
        F.col("Salary") * 1.10,
        2
    ).alias("RevisedSalary")
)

increment_df.show()

In [ ]:
# Aggregate functions
df.agg(
    F.count("*").alias("TotalEmployees"),
    F.count("Salary").alias("EmployeesWithSalary"),
    F.sum("Salary").alias("TotalSalary"),
    F.avg("Salary").alias("AverageSalary"),
    F.min("Salary").alias("MinimumSalary"),
    F.max("Salary").alias("MaximumSalary")
).show()

In [ ]:
department_summary = df.groupBy(
    "Department"
).agg(
    F.count("*").alias("EmployeeCount"),
    F.round(
        F.avg("Salary"), 2
    ).alias("AverageSalary"),
    F.sum("Salary").alias("TotalSalary"),
    F.max("Salary").alias("HighestSalary")
)

department_summary.orderBy("Department").show()

In [ ]:
df.groupBy(
    "Department",
    "City"
).agg(
    F.count("*").alias("EmployeeCount"),
    F.sum("Salary").alias("TotalSalary")
).show()

In [ ]:
df.select(
    F.countDistinct("Department").alias("UniqueDepartments")
).show()

In [ ]:
df.groupBy("Department").agg(
    F.collect_list(
        F.trim("Name")
    ).alias("Employees")
).show(truncate=False)

In [ ]:
# Null Handling Functions
df.filter(
    F.col("Salary").isNull()
).show()

In [ ]:
df.filter(
    F.col("Salary").isNotNull()
).show()

In [ ]:
coalesce_df = df.select(
    "Name",
    F.coalesce(
        F.col("City"),
        F.lit("Not Available")
    ).alias("City")
)

coalesce_df.show()

In [ ]:
filled_df = df.fillna({
    "Salary": 0.0,
    "City": "Unknown"
})

filled_df.show()

In [ ]:
from pyspark.sql.window import Window

department_window = Window.partitionBy("Department")

salary_df = df.withColumn(
    "DepartmentAverage",
    F.avg("Salary").over(department_window)
)

salary_df = salary_df.withColumn(
    "CleanSalary",
    F.coalesce(
        F.col("Salary"),
        F.col("DepartmentAverage")
    )
)

salary_df.select(
    "Name",
    "Department",
    "Salary",
    "DepartmentAverage",
    "CleanSalary"
).show()

In [ ]:
missing_data = [
    (101, "Amit", 75000.0, "Mumbai"),
    (102, "Neha", None, "Pune"),
    (103, None, 80000.0, "Delhi"),
    (104, "Priya", 65000.0, None),
    (105, "Ravi", None, None)
]

missing_df = spark.createDataFrame(
    missing_data,
    ["EmpID", "Name", "Salary", "City"]
)

missing_df.show()

In [ ]:
from pyspark.sql import functions as F
missing_df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in missing_df.columns
]).show()

In [ ]:
missing_df.dropna(
    how="any"
).show()

In [ ]:
missing_df.dropna(
    how="all"
).show()

In [ ]:
missing_df.dropna(
    subset=["EmpID", "Name"]
).show()

In [ ]:
filled_df = missing_df.fillna({
    "Name": "Unknown",
    "Salary": 0.0,
    "City": "Not Provided"
})

filled_df.show()

In [ ]:
average_salary = missing_df.agg(
    F.avg("Salary")
).first()[0]

imputed_df = missing_df.fillna({
    "Salary": average_salary
})

imputed_df.show()

In [ ]:
duplicate_data = [
    (101, "Amit", "IT", 75000),
    (102, "Neha", "HR", 55000),
    (101, "Amit", "IT", 75000),
    (103, "Rahul", "IT", 80000),
    (102, "Neha", "HR", 60000)
]

duplicate_df = spark.createDataFrame(
    duplicate_data,
    ["EmpID", "Name", "Department", "Salary"]
)

duplicate_df.show()

In [ ]:
unique_df = duplicate_df.distinct()

unique_df.show()

In [ ]:
unique_emp_df = duplicate_df.dropDuplicates(
    ["EmpID"]
)

unique_emp_df.show()

In [ ]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy(
    "EmpID"
).orderBy(
    F.col("Salary").desc()
)

ranked_df = duplicate_df.withColumn(
    "RowNum",
    F.row_number().over(window_spec)
)

deduplicated_df = ranked_df.filter(
    F.col("RowNum") == 1
).drop("RowNum")

deduplicated_df.orderBy("EmpID").show()

In [ ]:
type_data = [
    ("101", "Amit", "75000.50", "2024-01-15"),
    ("102", "Neha", "55000.75", "2023-06-20"),
    ("103", "Rahul", "80000.00", "2022-03-10")
]

type_df = spark.createDataFrame(
    type_data,
    ["EmpID", "Name", "Salary", "JoiningDate"]
)

type_df.printSchema()

In [ ]:
converted_df = type_df \
    .withColumn(
        "EmpID",
        F.col("EmpID").cast("integer")
    ) \
    .withColumn(
        "Salary",
        F.col("Salary").cast("decimal(12,2)")
    )

converted_df.printSchema()

In [ ]:
invalid_data = [
    (101, "Amit", 75000, "IT"),
    (102, "Neha", -5000, "HR"),
    (103, None, 80000, "IT"),
    (104, "Priya", 65000, "Unknown"),
    (105, "Ravi", 72000, "Finance")
]

invalid_df = spark.createDataFrame(
    invalid_data,
    ["EmpID", "Name", "Salary", "Department"]
)

In [ ]:
validated_df = invalid_df.withColumn(
    "ValidationStatus",
    F.when(
        F.col("EmpID").isNull(),
        "Invalid Employee ID"
    ).when(
        F.col("Name").isNull(),
        "Missing Name"
    ).when(
        F.col("Salary").isNull() |
        (F.col("Salary") < 0),
        "Invalid Salary"
    ).when(
        ~F.col("Department").isin(
            "IT", "HR", "Finance"
        ),
        "Invalid Department"
    ).otherwise("Valid")
)

validated_df.show(truncate=False)

In [ ]:
valid_records = validated_df.filter(
    F.col("ValidationStatus") == "Valid"
)

invalid_records = validated_df.filter(
    F.col("ValidationStatus") != "Valid"
)

print("VALID RECORDS")
valid_records.show()

print("INVALID RECORDS")
invalid_records.show()

In [20]:
valid_records.write \
    .mode("overwrite") \
    .parquet("output/clean/valid_employees")

invalid_records.write \
    .mode("overwrite") \
    .parquet("output/rejected/invalid_employees")

In [21]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

csv_schema = StructType([
    StructField("EmpID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Salary", DoubleType(), True),
    StructField("_corrupt_record", StringType(), True)
])

csv_df = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(csv_schema) \
    .csv("input/employees.csv")

In [22]:
csv_df.show()

+-----+-----+-------+-----------------+
|EmpID| Name| Salary|  _corrupt_record|
+-----+-----+-------+-----------------+
|  101| Amit|75000.0|             NULL|
|  102| Neha|55000.0|             NULL|
|  103|Rahul|   NULL|103,Rahul,INVALID|
|  104|Priya|65000.0|             NULL|
+-----+-----+-------+-----------------+

